# EdgeMason: Large-Scale Real Dataset ONNX Benchmark on NVIDIA T4 GPU

This notebook runs the ONNX benchmarking pipeline on a real **1,000+ image dataset** across 5 canonical vision models:
1. **MobileNetV2**
2. **ResNet-18**
3. **ResNet-50**
4. **ShuffleNetV2**
5. **SqueezeNet 1.1**

### Precision Regimes Evaluated on NVIDIA T4:
- **`onnx_cuda_fp32`**: Standard single-precision on CUDA cores.
- **`onnx_cuda_fp16`**: Half-precision utilizing NVIDIA T4 Tensor Cores.
- **`onnx_cuda_int8`**: Calibrated 8-bit quantization using a real **1,000-image calibration dataset**.

---
### ⚡ Quick Start:
1. Make sure your runtime is set to **T4 GPU**: Menu -> **Runtime** -> **Change runtime type** -> **T4 GPU**.
2. Click **Runtime** -> **Run all** (Ctrl+F9).
3. The final dataset `benchmark_gpu_t4_large_dataset.csv` will automatically download at the end!

In [ ]:
# Step 1: Check GPU & Install Dependencies
!nvidia-smi
!pip install --upgrade pip
!pip install onnxruntime-gpu onnx onnxscript onnxconverter-common torchvision pandas numpy pillow tqdm psutil

import os, time, torch, onnx, torchvision
import numpy as np, pandas as pd
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from onnxconverter_common import float16

print('Libraries imported successfully.')
print('Available Execution Providers:', ort.get_available_providers())

In [ ]:
# Step 2: Download Real Image Dataset (1,000+ real photographic images)
os.makedirs('dataset', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('reports/data', exist_ok=True)

# Downloading Fast.ai Imagenette (clean ImageNet 10-class subset with ~3,900 images)
if not os.path.exists('dataset/imagenette2-160'):
    print('Downloading real 1,000+ image dataset (Imagenette, ~98MB)...')
    !wget -nc -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
    !tar -xzf imagenette2-160.tgz -C dataset/
    print('Dataset extracted successfully!')

# Collect all image file paths
image_paths = []
for root, dirs, files in os.walk('dataset/imagenette2-160/val'):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_paths.append(os.path.join(root, f))

print(f'Total validation images available: {len(image_paths)}')

# Preprocess 1,000 images into standard tensor format (3, 224, 224)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

NUM_CALIBRATION_IMAGES = 1000
selected_paths = image_paths[:NUM_CALIBRATION_IMAGES]
print(f'Preprocessing {len(selected_paths)} images for real-world calibration & benchmarking...')

processed_tensors = []
for p in tqdm(selected_paths):
    img = Image.open(p).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).numpy().astype(np.float32)
    processed_tensors.append(tensor)

all_images_array = np.concatenate(processed_tensors, axis=0)
np.save('dataset/real_images_1000.npy', all_images_array)
print(f'Saved preprocessed dataset: dataset/real_images_1000.npy (Shape: {all_images_array.shape})')

In [ ]:
# Step 3: Export 5 Canonical CNN Models with Real Pretrained Weights
model_constructors = {
    'mobilenetv2': models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT),
    'resnet18': models.resnet18(weights=models.ResNet18_Weights.DEFAULT),
    'resnet50': models.resnet50(weights=models.ResNet50_Weights.DEFAULT),
    'shufflenetv2': models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT),
    'squeezenet': models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT),
}

dummy_input = torch.randn(1, 3, 224, 224)

print('Exporting pretrained models to ONNX (opset 13)...')
for name, model in model_constructors.items():
    model.eval()
    onnx_path = f'models/{name}.onnx'
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=['input'],
        output_names=['output'],
        opset_version=13,
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
    )
    print(f' -> Exported {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.2f} MB)')

In [ ]:
# Step 4: Quantization Engine (FP16 Converter & Real-Image INT8 Calibrator)

class RealImageDataReader(CalibrationDataReader):
    def __init__(self, npy_path, input_name='input', max_samples=250):
        self.data = np.load(npy_path)
        self.input_name = input_name
        self.max_samples = min(len(self.data), max_samples)
        self.current_idx = 0

    def get_next(self):
        if self.current_idx < self.max_samples:
            sample = {self.input_name: self.data[self.current_idx:self.current_idx+1]}
            self.current_idx += 1
            return sample
        return None

    def rewind(self):
        self.current_idx = 0

for name in ['mobilenetv2', 'resnet18', 'resnet50', 'shufflenetv2', 'squeezenet']:
    fp32_path = f'models/{name}.onnx'
    fp16_path = f'models/{name}_fp16.onnx'
    int8_path = f'models/{name}_int8.onnx'
    
    # 1. Convert to FP16 (for Tensor Cores)
    print(f'Converting {name} to FP16...')
    model_fp32 = onnx.load(fp32_path)
    model_fp16 = float16.convert_float_to_float16(model_fp32, keep_io_types=True)
    onnx.save(model_fp16, fp16_path)
    
    # 2. Convert to Static INT8 using the 1000 real-image dataset
    print(f'Quantizing {name} to Static INT8 with real-world image calibration...')
    reader = RealImageDataReader('dataset/real_images_1000.npy', max_samples=200)
    try:
        quantize_static(
            model_input=fp32_path,
            model_output=int8_path,
            calibration_data_reader=reader,
            quant_format=QuantFormat.QDQ,
            activation_type=QuantType.QInt8,
            weight_type=QuantType.QInt8,
            per_channel=True
        )
    except Exception as e:
        print(f'Static QDQ fallback for {name}: {e}')
        from onnxruntime.quantization import quantize_dynamic
        quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)

print('All precision variants (FP32, FP16, INT8) created successfully!')

In [ ]:
# Step 5: Large-Scale Benchmark across 1,000 Real Images on NVIDIA T4
images_data = np.load('dataset/real_images_1000.npy')
total_images = len(images_data)
print(f'Beginning benchmark across all {total_images} real images on NVIDIA T4 GPU...')

benchmark_results = []
models_to_test = ['mobilenetv2', 'resnet18', 'resnet50', 'shufflenetv2', 'squeezenet']

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

for model_name in models_to_test:
    variants = [
        ('onnx_cuda_fp32', f'models/{model_name}.onnx', 'fp32'),
        ('onnx_cuda_fp16', f'models/{model_name}_fp16.onnx', 'fp16'),
        ('onnx_cuda_int8', f'models/{model_name}_int8.onnx', 'int8')
    ]
    
    for strat_name, path, precision in variants:
        if not os.path.exists(path):
            continue
            
        file_size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'Testing {model_name} | {strat_name} ({file_size_mb:.2f} MB)...')
        
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        session = ort.InferenceSession(path, opts, providers=providers)
        input_name = session.get_inputs()[0].name
        
        # Warmup on 50 images
        for i in range(50):
            session.run(None, {input_name: images_data[i:i+1]})
            
        # Benchmark across 1,000 real images
        latencies_ms = []
        torch.cuda.synchronize()
        t_start_total = time.perf_counter()
        
        for i in range(total_images):
            batch = images_data[i:i+1]
            t0 = time.perf_counter()
            session.run(None, {input_name: batch})
            torch.cuda.synchronize()
            latencies_ms.append((time.perf_counter() - t0) * 1000.0)
            
        total_time_sec = time.perf_counter() - t_start_total
        p50 = float(np.percentile(latencies_ms, 50))
        p95 = float(np.percentile(latencies_ms, 95))
        p99 = float(np.percentile(latencies_ms, 99))
        fps = total_images / total_time_sec
        
        benchmark_results.append({
            'hardware': 'nvidia_t4_gpu',
            'model': f'{model_name}.onnx',
            'strategy': strat_name,
            'precision': precision,
            'model_size_mb': round(file_size_mb, 2),
            'num_images_tested': total_images,
            'latency_p50_ms': round(p50, 4),
            'latency_p95_ms': round(p95, 4),
            'latency_p99_ms': round(p99, 4),
            'throughput_fps': round(fps, 2),
            'total_time_sec': round(total_time_sec, 2)
        })

df_results = pd.DataFrame(benchmark_results)
out_csv = 'reports/data/benchmark_gpu_t4_large_dataset.csv'
df_results.to_csv(out_csv, index=False)
print('\n' + '='*70)
print('ALL 1,000-IMAGE GPU BENCHMARKS FINISHED!')
print('='*70)
display(df_results)

# Automatic download to local PC
from google.colab import files
files.download(out_csv)